# Exercise 2.1: Basic Neural Network Training with PyTorch

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/exercises/exercise_2_1_basic_neural_network_tabular.ipynb)

This exercise follows the first deep learning lecture. The goal is to train a small neural network without going too deep into the mathematics.

You will practice:

- creating a small tabular dataset
- splitting and scaling features
- converting data to PyTorch tensors
- building a neural network with `torch.nn.Module`
- training with `CrossEntropyLoss`, backpropagation, and an optimizer
- evaluating accuracy, loss curves, and a confusion matrix

The example is a small urban tabular classification task. Each row describes one sampled city location with simple attributes. The model predicts the area type.


## 0. Colab setup

Run this first in Colab. PyTorch is often preinstalled, but the command below makes the notebook self-contained.


In [ ]:
!pip -q install torch scikit-learn pandas matplotlib seaborn

In [ ]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


## 1. Create a small tabular dataset

The dataset is synthetic but geospatially motivated. Each row represents a sampled location in a city. The features are simple local attributes:

- `distance_to_center_km`
- `road_density`
- `green_share`
- `transit_access`
- `building_density`
- `night_light`

The target label is `area_type`.


In [ ]:
def make_city_tabular_data(seed=42):
    rng = np.random.default_rng(seed)
    rows = []

    specs = [
        {
            "area_type": "Residential",
            "n": 90,
            "distance": (3.2, 0.8),
            "road": (4.5, 0.7),
            "green": (0.28, 0.08),
            "transit": (0.48, 0.12),
            "building": (0.55, 0.10),
            "light": (0.45, 0.10),
        },
        {
            "area_type": "Commercial center",
            "n": 80,
            "distance": (0.9, 0.45),
            "road": (7.2, 0.8),
            "green": (0.10, 0.04),
            "transit": (0.86, 0.08),
            "building": (0.82, 0.08),
            "light": (0.88, 0.07),
        },
        {
            "area_type": "Green / recreation",
            "n": 75,
            "distance": (2.6, 0.9),
            "road": (2.0, 0.55),
            "green": (0.74, 0.09),
            "transit": (0.30, 0.10),
            "building": (0.18, 0.08),
            "light": (0.22, 0.08),
        },
        {
            "area_type": "Industrial / logistics",
            "n": 70,
            "distance": (4.8, 0.9),
            "road": (6.3, 0.9),
            "green": (0.16, 0.06),
            "transit": (0.36, 0.11),
            "building": (0.70, 0.10),
            "light": (0.63, 0.12),
        },
    ]

    for spec in specs:
        for _ in range(spec["n"]):
            rows.append(
                {
                    "distance_to_center_km": max(0.05, rng.normal(*spec["distance"])),
                    "road_density": max(0.0, rng.normal(*spec["road"])),
                    "green_share": np.clip(rng.normal(*spec["green"]), 0, 1),
                    "transit_access": np.clip(rng.normal(*spec["transit"]), 0, 1),
                    "building_density": np.clip(rng.normal(*spec["building"]), 0, 1),
                    "night_light": np.clip(rng.normal(*spec["light"]), 0, 1),
                    "area_type": spec["area_type"],
                }
            )

    return pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)

city_df = make_city_tabular_data(SEED)
display(city_df.head())
print(city_df["area_type"].value_counts())


In [ ]:
feature_columns = [
    "distance_to_center_km",
    "road_density",
    "green_share",
    "transit_access",
    "building_density",
    "night_light",
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()
for ax, col in zip(axes, feature_columns):
    sns.kdeplot(data=city_df, x=col, hue="area_type", fill=False, common_norm=False, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


### Student task

Before training the neural network, inspect the feature plots:

- Which features separate the classes clearly?
- Which classes look similar?
- Which feature would be hard to use alone?


## 2. Split, scale, and convert to tensors

Neural networks train more smoothly when numerical features are scaled. We use `StandardScaler` so each feature has roughly mean 0 and standard deviation 1.

`CrossEntropyLoss` expects class labels as integer IDs: `0`, `1`, `2`, ...


In [ ]:
X = city_df[feature_columns].values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(city_df["area_type"])
class_names = label_encoder.classes_

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.35,
    random_state=SEED,
    stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
val_ds = TensorDataset(X_val_tensor, y_val_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

print("Classes:", list(class_names))
print("Train / val / test:", len(train_ds), len(val_ds), len(test_ds))
print("One batch shapes:")
xb, yb = next(iter(train_loader))
print("X:", xb.shape, "y:", yb.shape)


## 3. Define a small neural network

This model is a multi-layer perceptron (MLP):

1. a linear layer transforms the input features into hidden activations
2. `ReLU` adds non-linearity
3. another linear layer produces one score per class

The output values are called logits. We do not add `softmax` in the model because `CrossEntropyLoss` applies the correct transformation internally.


In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.network(x)

model = TabularMLP(
    input_dim=len(feature_columns),
    hidden_dim=16,
    output_dim=len(class_names),
).to(DEVICE)

print(model)


In [ ]:
with torch.no_grad():
    xb, yb = next(iter(train_loader))
    logits = model(xb.to(DEVICE))

print("Input batch shape: ", xb.shape)
print("Output logits shape:", logits.shape)
print("First row of logits:", logits[0].cpu().numpy().round(3))


## 4. Train the network

The training loop follows the lecture summary:

1. forward pass: compute predictions
2. loss: compare predictions with labels
3. backprop: compute gradients with `loss.backward()`
4. update: change parameters with `optimizer.step()`

PyTorch handles the gradient calculations. You only need to write the training steps in the right order.


In [ ]:
def evaluate(model, data_loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = loss_fn(logits, yb)

            total_loss += loss.item() * len(xb)
            predictions = logits.argmax(dim=1)
            correct += (predictions == yb).sum().item()
            total += len(xb)

    return total_loss / total, correct / total

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

history = []
EPOCHS = 120

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * len(xb)
        correct_train += (logits.argmax(dim=1) == yb).sum().item()
        total_train += len(xb)

    train_loss = total_train_loss / total_train
    train_acc = correct_train / total_train
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
        }
    )

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"epoch {epoch:03d} | "
            f"train loss {train_loss:.3f} | val loss {val_loss:.3f} | "
            f"train acc {train_acc:.3f} | val acc {val_acc:.3f}"
        )

history_df = pd.DataFrame(history)


## 5. Plot the learning curves

The loss curve shows whether training is still improving. The validation curve helps identify overfitting.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(data=history_df, x="epoch", y="train_loss", label="train", ax=axes[0])
sns.lineplot(data=history_df, x="epoch", y="val_loss", label="validation", ax=axes[0])
axes[0].set_title("Loss during training")
axes[0].set_ylabel("Cross-entropy loss")

sns.lineplot(data=history_df, x="epoch", y="train_accuracy", label="train", ax=axes[1])
sns.lineplot(data=history_df, x="epoch", y="val_accuracy", label="validation", ax=axes[1])
axes[1].set_title("Accuracy during training")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()


### Student task

Change one setting and rerun the training cells:

- `hidden_dim`: try `4`, `16`, and `64`
- `lr`: try `0.001`, `0.01`, and `0.05`
- `EPOCHS`: try `30` and `200`

What changes in the loss and validation accuracy curves?


## 6. Evaluate on the test set

Use the test set only after you have chosen your model settings. This gives a more honest estimate of performance on unseen data.


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"Test loss:     {test_loss:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

model.eval()
all_logits = []
with torch.no_grad():
    for xb, _ in test_loader:
        all_logits.append(model(xb.to(DEVICE)).cpu())

logits_test = torch.cat(all_logits, dim=0)
y_pred = logits_test.argmax(dim=1).numpy()

print(classification_report(y_test, y_pred, target_names=class_names))

fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=class_names,
    xticks_rotation=35,
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Neural network confusion matrix")
plt.tight_layout()
plt.show()


## 7. Predict one new location

A trained model can be used for new tabular observations. The important detail is that the same scaler must be used before prediction.


In [ ]:
new_location = pd.DataFrame(
    [
        {
            "distance_to_center_km": 0.7,
            "road_density": 7.8,
            "green_share": 0.08,
            "transit_access": 0.91,
            "building_density": 0.86,
            "night_light": 0.92,
        }
    ]
)

new_scaled = scaler.transform(new_location[feature_columns])
new_tensor = torch.tensor(new_scaled, dtype=torch.float32).to(DEVICE)

model.eval()
with torch.no_grad():
    logits = model(new_tensor)
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]

prediction_table = pd.DataFrame(
    {
        "area_type": class_names,
        "probability": probabilities,
    }
).sort_values("probability", ascending=False)

display(new_location)
display(prediction_table)


## 8. Short reflection

Answer these questions in two or three sentences each:

- What are the inputs, hidden layers, output logits, loss, and optimizer in this notebook?
- Why do we scale tabular features before training?
- Why do we keep a validation set separate from the test set?
- When did the neural network clearly outperform or fail to outperform simpler machine learning methods from Exercise 1.2?

The point is not to memorize PyTorch syntax. The point is to understand the training workflow: forward pass, loss, backpropagation, and parameter update.
